Now we have taken the raw model and fine tunned the model with pahrma data, so it can understand the pharma related patterns.
This is custome pretraining. now it has domain specific knowledge but not ready fro the Q&A that means conversation.

it just good for generate something but not for conversational AI.

After IFT --> converstion ready but not for prefernce



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# Unsloth 3-Stage Pharma Fine-Tuning Pipeline
# Proper Flow:
# Stage 1 Adapter -> Merge -> Load Merged for Stage 2
# Stage 2 Adapter -> Merge -> Load Merged for Stage 3
# Stage 3 DPO Adapter -> Final Merge
# ============================================================

# -------------------------
# 1. Install libraries
# -------------------------
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.7.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
unsloth 2026.7.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.


In [ ]:
# -------------------------
# 2. Imports
# -------------------------
import os
import re
import gc
import time
import json
import unicodedata
import warnings
from typing import List, Dict, Any

warnings.filterwarnings("ignore")

import torch
import fitz  # PyMuPDF
from datasets import Dataset, load_dataset
from huggingface_hub import login, whoami, hf_hub_download
from transformers import TextStreamer
from google.colab import userdata
import unsloth  # keep this import early
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

try:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
    print("DPO patch applied.")
except Exception as e:
    print("DPO patch skipped:", repr(e))

from trl import DPOTrainer, DPOConfig

assert torch.cuda.is_available(), "GPU not found. In Colab: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

DPO patch applied.
GPU: Tesla T4


In [ ]:
# ============================================================
# Global configuration
# ============================================================
# Keep all important parameters in one place.
# This makes the notebook easier to debug, reproduce, and productionize.

from dataclasses import dataclass, asdict

@dataclass
class Config:
  BASE_MODEL_NAME = "unsloth/tinyllama-bnb-4bit"
  DATASET_SIZE = 100
  MAX_SEQ_LENGTH = 350
  MIN_CHARS_PER_PARAGRAPH = 80
  SEED = 42

  LORA_R = 16
  LORA_ALPHA = 32
  LORA_DROPOUT = 0

  BATCH_SIZE = 1
  GRAD_ACCUM_STEPS = 8
  WARMUP_STEPS = 5
  LOGGING_STEPS = 1

  STAGE1_MAX_STEPS = 60
  STAGE2_MAX_STEPS = 120
  STAGE3_MAX_STEPS = 120

  STAGE1_LR = 2e-4
  STAGE2_LR = 1e-4
  STAGE3_LR = 5e-5

  DPO_BETA = 0.1

  OUTPUT_ROOT = "/content/unsloth_yes_merge_reload_outputs"
  STAGE1_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage1_non_instruction_adapter"
  STAGE1_MERGED_DIR  = f"{OUTPUT_ROOT}/stage1_non_instruction_merged_model"

  STAGE2_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage2_instruction_adapter"
  STAGE2_MERGED_DIR  = f"{OUTPUT_ROOT}/stage2_instruction_merged_model"

  STAGE3_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage3_dpo_adapter"
  FINAL_MERGED_DIR   = f"{OUTPUT_ROOT}/stage3_dpo_final_merged_model"

  repo_id = "Suresh18/yesbank-marquee-assignment"
  data_files = "non_instruction/train-00000-of-00001.parquet"
  model_repo_id = f"Suresh18/yesbank-marquee-assignment"

config = Config()

In [ ]:
for path in [
    config.OUTPUT_ROOT,
    config.STAGE1_ADAPTER_DIR,
    config.STAGE1_MERGED_DIR,
    config.STAGE2_ADAPTER_DIR,
    config.STAGE2_MERGED_DIR,
    config.STAGE3_ADAPTER_DIR,
    config.FINAL_MERGED_DIR,
]:
    os.makedirs(path, exist_ok=True)

In [ ]:
stage1_dataset = load_dataset(
                        path=config.repo_id,
                        data_files=config.data_files,
                        revision="v4.0"  # This targets your v4 branch
                        )

In [ ]:
stage1_dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 287
    })
})

In [ ]:
# -------------------------
# 5. Helper functions
# -------------------------
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()


def train_and_measure(trainer, stage_name: str):
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()
    result = trainer.train()
    torch.cuda.synchronize()

    train_time = round(time.time() - start_time, 2)
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    return result


def load_unsloth_model_with_lora(model_name_or_path: str, config: dict):
    """
    Loads a base or merged model in 4-bit and attaches a fresh LoRA adapter.
    This is used at each stage:
    - Stage 1 loads BASE_MODEL_NAME
    - Stage 2 loads STAGE1_MERGED_DIR
    - Stage 3 loads STAGE2_MERGED_DIR
    """

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=config.MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=config.LORA_R,
        lora_alpha=config.LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=config.LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=config.SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer

def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str, stage_name: str):
    """
    Saves LoRA adapter separately and also saves a merged standalone model.
    The merged model becomes the starting point for the next stage.
    """

    print(f"\nSaving {stage_name} adapter...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"{stage_name} adapter saved to:", adapter_dir)

    print(f"\nMerging {stage_name} adapter with base model...")
    FastLanguageModel.for_training(model)

    model.save_pretrained_merged(
        merged_dir,
        tokenizer,
        save_method="merged_16bit",
    )

    print(f"{stage_name} merged model saved to:", merged_dir)

def build_instruction_prompt(instruction: str, input_text: str = "") -> str:
    instruction = str(instruction).strip()
    input_text = str(input_text).strip()

    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

    return f"### Instruction:\n{instruction}\n\n### Response:\n"

def generate_answer(model, tokenizer, instruction: str, input_text: str = "", max_new_tokens: int = 150):
    FastLanguageModel.for_inference(model)

    prompt = build_instruction_prompt(instruction, input_text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_tokens = inputs["input_ids"].shape[-1]
    generated_tokens = output[0][input_tokens:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

In [ ]:
# ============================================================
# STAGE 1: Non-instruction continued pretraining
# ============================================================

print("\n==============================")
print("STAGE 1: PDF RAW TEXT TRAINING")
print("==============================")

stage1_model, tokenizer = load_unsloth_model_with_lora(config.BASE_MODEL_NAME, config)

FastLanguageModel.for_training(stage1_model)


STAGE 1: PDF RAW TEXT TRAINING
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048, padding_idx=0)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Line

In [ ]:
stage1_dataset['train']

Dataset({
    features: ['text'],
    num_rows: 287
})

In [ ]:
stage1_config = SFTConfig(
    output_dir=f"{config.OUTPUT_ROOT}/stage1_logs",

    max_steps=config.STAGE1_MAX_STEPS,
    per_device_train_batch_size=config.BATCH_SIZE,
    gradient_accumulation_steps=config.GRAD_ACCUM_STEPS,

    learning_rate=config.STAGE1_LR,
    warmup_steps=config.WARMUP_STEPS,

    logging_steps=config.LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=config.MAX_SEQ_LENGTH,
    packing=True,

    seed=config.SEED,
)

stage1_trainer = SFTTrainer(
    model=stage1_model,
    processing_class=tokenizer,
    train_dataset=stage1_dataset['train'].select(range(150)),
    args=stage1_config,
)

train_and_measure(stage1_trainer, "STAGE 1 - NON-INSTRUCTION PDF TRAINING")

save_adapter_and_merge(
    model=stage1_model,
    tokenizer=tokenizer,
    adapter_dir=config.STAGE1_ADAPTER_DIR,
    merged_dir=config.STAGE1_MERGED_DIR,
    stage_name="Stage 1",
)

# del stage1_trainer
# del stage1_model
# clear_gpu_memory()

Unsloth: Sample packing skipped (custom data collator detected).


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/150 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 150 | Num Epochs = 4 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


Step,Training Loss
1,4.808400
2,4.467100
3,4.698900
4,4.375500
5,4.247300
6,4.865600
7,4.384100
8,3.982200
9,4.741800
10,3.931300



STAGE 1 - NON-INSTRUCTION PDF TRAINING RESULTS
Train time/sec: 152.3
Peak allocated VRAM/GB: 3.207
Peak reserved VRAM/GB: 3.33

Saving Stage 1 adapter...
Stage 1 adapter saved to: /content/unsloth_yes_merge_reload_outputs/stage1_non_instruction_adapter

Merging Stage 1 adapter with base model...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:03<00:00, 63.23s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_yes_merge_reload_outputs/stage1_non_instruction_merged_model`
Stage 1 merged model saved to: /content/unsloth_yes_merge_reload_outputs/stage1_non_instruction_merged_model


In [ ]:
from huggingface_hub import HfApi
HF_TOKEN = userdata.get("HF_TOKEN_WRITE")
login(token=HF_TOKEN)
print("Name:", whoami()['fullname'])
api = HfApi()
api.create_repo(repo_id = config.model_repo_id, repo_type="model", exist_ok=True)

Name: Veera Venkata Suresh Grandhi


RepoUrl('https://huggingface.co/Suresh18/yesbank-marquee-assignment', endpoint='https://huggingface.co', repo_type='model', repo_id='Suresh18/yesbank-marquee-assignment')

In [ ]:
# ==========================================
# STAGE 1: NON-INSTRUCTION TUNING FOLDERS
# ==========================================
print("\n--- Uploading Stage 1: Non-Instruction ---")
# Uploads LoRA to: /non_instruction/lora/
api.upload_folder(
    folder_path="/content/unsloth_yes_merge_reload_outputs/stage1_non_instruction_adapter",
    repo_id=config.model_repo_id,
    path_in_repo="non_instruction/lora"
)

# Uploads Merged to: /non_instruction/merged/
api.upload_folder(
    folder_path="/content/unsloth_yes_merge_reload_outputs/stage1_non_instruction_merged_model",
    repo_id=config.model_repo_id,
    path_in_repo="non_instruction/merged"
)


--- Uploading Stage 1: Non-Instruction ---


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n_adapter/tokenizer.model: 100%|##########|  500kB /  500kB            

  ...adapter_model.safetensors:  15%|#4        | 7.37MB / 50.5MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ged_model/tokenizer.model: 100%|##########|  500kB /  500kB            

  ...d_model/model.safetensors:   2%|2         | 47.9MB / 2.20GB            

CommitInfo(commit_url='https://huggingface.co/Suresh18/yesbank-marquee-assignment/commit/e2f2778e38ba570032da749a428a480849707f98', commit_message='Upload folder using huggingface_hub', commit_description='', oid='e2f2778e38ba570032da749a428a480849707f98', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Suresh18/yesbank-marquee-assignment', endpoint='https://huggingface.co', repo_type='model', repo_id='Suresh18/yesbank-marquee-assignment'), pr_revision=None, pr_num=None)

**The Strategy:**

Use "Few-Shot" PromptingTo make a non-instruction model give you a specific answer, you must provide a pattern of examples (few-shot prompting) so the model naturally understands that the next logical sentence it needs to generate is the answer.

  **Instead of writing :** "What is the capital of France?"

  **You must format your prompt text like this:**

  "The capital of Germany is Berlin. The capital of Spain is Madrid. The capital of France is"The model will read that text pattern and naturally predict the next token:  Paris.

  

In [ ]:
# 1. Loading the model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/unsloth_yes_merge_reload_outputs/stage1_non_instruction_merged_model", # Path to your Colab folder or HF repo path
    max_seq_length = 512,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)


# 2. Input the exact pattern prompt
few_shot_prompt = """Card Name: YES BANK POP-CLUB Credit Card
Joining Fee Waiver: Yes, Zero Joining Fee by default.
Annual Fee Waiver: Yes, via specific spend conditions.
---
Card Name: YES FIRST Business Credit Card
Joining Fee Waiver: Yes, waived on retail spends of INR 20,000 within 1 month.
Annual Fee Waiver: Yes, waived on retail spends of INR 3,00,000 within 12 months.
---
Card Name: YES BANK MARQUÉE Credit Card
Joining Fee Waiver: No, one-time non-waivable fee of INR 9,999 + GST applies.
Annual Fee Waiver:"""

# 3. Tokenize and stream the model's text continuation
inputs = tokenizer([few_shot_prompt], return_tensors = "pt").to("cuda")
text_streamer = TextStreamer(tokenizer)

_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 40, # Keeps it short to just finish the final line
    temperature = 0.1     # Keeps the answer highly accurate and grounded
)

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
<s> Card Name: YES BANK POP-CLUB Credit Card
Joining Fee Waiver: Yes, Zero Joining Fee by default.
Annual Fee Waiver: Yes, via specific spend conditions.
---
Card Name: YES FIRST Business Credit Card
Joining Fee Waiver: Yes, waived on retail spends of INR 20,000 within 1 month.
Annual Fee Waiver: Yes, waived on retail spends of INR 3,00,000 within 12 months.
---
Card Name: YES BANK MARQUÉE Credit Card
Joining Fee Waiver: No, one-time non-waivable fee of INR 9,999 + GST applies.
Annual Fee Waiver: Yes, via specific spend conditions.
---
Card Na

In [ ]:
# del stage1_trainer
# del stage1_model
# clear_gpu_memory()